In [19]:
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
import random
from Functions.Utils import *
from Functions.Graphs import *
random.seed(101)
mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

path = r'Datasets\DEVRT\NISSAN LEAF\20230421_NISSAN_DONOSTIA_ULIA_056.csv'
df = pd.read_csv(path)
df.fillna(0, inplace=True)

pwr = (df['Motor Pwr(w)'].values)
spd = (df['speed'].values)
rpm = (df['rpm'].values)
trq = (df['Torque Nm'].values)
elv = (df['elv_spy'].values)
lat = (df['latitude'].values)
lon = (df['longitude'].values)
alt = (df['altitude'].values)
df = pd.DataFrame({'speed': spd, 'rpm': rpm, 'torque': trq, 'elv': elv,'lat': lat, 'lon': lon, 'alt': alt, 'power': pwr,})
df = pd.DataFrame({'speed': spd, 'rpm': rpm, 'torque': trq, 'elv': elv, 'power': pwr,})
df = pd.DataFrame({'speed': spd, 'rpm': rpm, 'torque': trq, 'power': pwr,})

def normalizar_3d(array_3d, scaler, num_cols):
    N, T, F = array_3d.shape
    scaled = scaler.transform(array_3d.reshape(-1, num_cols))
    return scaled.reshape(N, T, F)

def PrepareDataframe(df,Np=2):
    pwr = df.iloc[:,-1].values
    df1 = df.iloc[:-Np,:-1].copy()
    for i in range(Np):
        if i < Np:
            sig = pwr[i:-Np+i]
            df1[f'power{i}'] = sig
        if i+1 == Np:
            sig = pwr[Np:]
            df1[f'power{i+1}'] = sig
    return df1

def SplitData(df,Np,time_steps, increase=1):
    df = PrepareDataframe(df,Np)
    df = pd.concat([df] * increase, ignore_index=True)
    X = df.iloc[:,:-(Np+1)].values
    y = df.iloc[:,-(Np+1):].values

    Xs, ys = [], []
    for i in range(len(X) - time_steps + 1):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i:(i + time_steps)])

    Xs, ys = np.array(Xs), np.array(ys)

    return Xs, ys

def DataTrainSplit(df,Np,time_steps, increase=1):
    df = PrepareDataframe(df,Np)
    df = pd.concat([df] * increase, ignore_index=True)
    X = df.iloc[:,:-(Np+1)].values
    y = df.iloc[:,-(Np+1):].values

    Xs, ys = [], []
    for i in range(len(X) - time_steps + 1):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i:(i + time_steps)])

    Xs, ys = np.array(Xs), np.array(ys)
    
    n = len(Xs)
    x_train = Xs[0:int(n*0.7)]
    x_test = Xs[int(n*0.7):int(n*0.9)]
    x_val = Xs[int(n*0.9):]
    y_train = ys[0:int(n*0.7)]
    y_test = ys[int(n*0.7):int(n*0.9)]
    y_val = ys[int(n*0.9):]

    return x_train, x_test, x_val, y_train, y_test, y_val

In [20]:
path = r'Datasets\DEVRT\NISSAN LEAF'
samples = os.listdir(path)

for i,sample in enumerate(samples):
    df = pd.read_csv(os.path.join(path,sample))
    pwr = (df['Motor Pwr(w)'].values)
    spd = (df['speed'].values)
    rpm = (df['rpm'].values)
    trq = (df['Torque Nm'].values)
    elv = (df['elv_spy'].values)
    lat = (df['latitude'].values)
    lon = (df['longitude'].values)
    alt = (df['altitude'].values)
    df = pd.DataFrame({'speed': spd, 'rpm': rpm, 'torque': trq, 'elv': elv,'lat': lat, 'lon': lon, 'alt': alt, 'power': pwr,})

    if i == 0:
        df1 = df.copy()
    else:
        df1 = pd.concat([df1,df])


In [21]:
class PowerPredictorKerasStyleLSTM(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=128, num_layers=3, output_dim=1):
        super(PowerPredictorKerasStyleLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # O parâmetro dropout exige num_layers > 1
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.1)
        self.linear = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.linear(out)


In [22]:
Np=5
time_steps=20
increase=10
x_train, x_test, x_val, y_train, y_test, y_val = DataTrainSplit(df1,Np,time_steps, increase)


num_features = x_train.shape[2]
num_labels = y_train.shape[2]
time_steps = x_train.shape[1]

# Garante que y tenha 3 dimensões [N, 3, 1]
if y_train.ndim == 2:
    y_train = np.expand_dims(y_train, axis=-1)
    y_test = np.expand_dims(y_test, axis=-1)
    y_val = np.expand_dims(y_val, axis=-1)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# Ajusta os scalers usando apenas o conjunto de treino (Boas práticas acadêmicas)
scaler_X.fit(x_train.reshape(-1, num_features))
scaler_y.fit(y_train.reshape(-1, num_labels))

x_train_scaled = normalizar_3d(x_train, scaler_X, num_features)
x_test_scaled  = normalizar_3d(x_test, scaler_X, num_features)
x_val_scaled   = normalizar_3d(x_val, scaler_X, num_features)

y_train_scaled = normalizar_3d(y_train, scaler_y, num_labels)
y_test_scaled  = normalizar_3d(y_test, scaler_y, num_labels)
y_val_scaled   = normalizar_3d(y_val, scaler_y, num_labels)

# --- 2. Criação dos DataLoaders PyTorch ---
train_dataset = TensorDataset(torch.tensor(x_train_scaled, dtype=torch.float32), torch.tensor(y_train_scaled, dtype=torch.float32))
val_dataset   = TensorDataset(torch.tensor(x_val_scaled, dtype=torch.float32), torch.tensor(y_val_scaled, dtype=torch.float32))

# Ativamos shuffle apenas no treino. Validação precisa manter a ordem temporal.
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=512, shuffle=False, pin_memory=True)

# --- 3. Instanciação da Rede, Critério e Otimizador ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lstm = PowerPredictorKerasStyleLSTM(input_dim=num_features, hidden_dim=128, num_layers=2, output_dim=num_labels).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(lstm.parameters(), lr=0.005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
scaler_amp = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

x_train_scaled = normalizar_3d(x_train, scaler_X, num_features)
x_test_scaled  = normalizar_3d(x_test, scaler_X, num_features)
x_val_scaled   = normalizar_3d(x_val, scaler_X, num_features)

y_train_scaled = normalizar_3d(y_train, scaler_y, num_labels)
y_test_scaled  = normalizar_3d(y_test, scaler_y, num_labels)
y_val_scaled   = normalizar_3d(y_val, scaler_y, num_labels)

# --- 2. Criação dos DataLoaders PyTorch ---
train_dataset = TensorDataset(torch.tensor(x_train_scaled, dtype=torch.float32), torch.tensor(y_train_scaled, dtype=torch.float32))
val_dataset   = TensorDataset(torch.tensor(x_val_scaled, dtype=torch.float32), torch.tensor(y_val_scaled, dtype=torch.float32))

# Ativamos shuffle apenas no treino. Validação precisa manter a ordem temporal.
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=512, shuffle=False, pin_memory=True)

C:\Users\Claudio\AppData\Roaming\Python\Python310\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [23]:
# --- 3. Instanciação da Rede, Critério e Otimizador ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lstm = PowerPredictorKerasStyleLSTM(input_dim=num_features, hidden_dim=128, num_layers=3, output_dim=num_labels).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(lstm.parameters(), lr=0.005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
scaler_amp = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

# --- 4. Loop de Treinamento e Validação Intercalado ---
EPOCHS = 50
print(f"Iniciando ciclo de lstmagem na GPU: {device}\n")

for epoch in range(EPOCHS):
    # FASE DE TREINO
    lstm.train()
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device, non_blocking=True), batch_y.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            predictions = lstm(batch_X)
            loss = criterion(predictions, batch_y)
        
        scaler_amp.scale(loss).backward()
        scaler_amp.step(optimizer)
        scaler_amp.update()
        
        train_loss += loss.item() * batch_X.size(0)
    
    total_train_loss = train_loss / len(train_loader.dataset)
    
    # FASE DE VALIDAÇÃO (Ocorre ao fim de cada época)
    lstm.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            predictions = lstm(batch_X)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item() * batch_X.size(0)
            
    total_val_loss = val_loss / len(val_loader.dataset)
    
    # Log de acompanhamento acadêmico
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Época [{epoch+1:02d}/{EPOCHS}] | MSE Treino: {total_train_loss:.6f} | MSE Validação: {total_val_loss:.6f}")

print("\nTreinamento e validação concluídos!")

C:\Users\Claudio\AppData\Roaming\Python\Python310\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Iniciando ciclo de lstmagem na GPU: cuda

Época [01/50] | MSE Treino: 0.032984 | MSE Validação: 0.029114
Época [05/50] | MSE Treino: 0.022191 | MSE Validação: 0.018024
Época [10/50] | MSE Treino: 0.007555 | MSE Validação: 0.006350
Época [15/50] | MSE Treino: 0.005755 | MSE Validação: 0.004846
Época [20/50] | MSE Treino: 0.004912 | MSE Validação: 0.004003
Época [25/50] | MSE Treino: 0.004397 | MSE Validação: 0.003565
Época [30/50] | MSE Treino: 0.004098 | MSE Validação: 0.003240
Época [35/50] | MSE Treino: 0.003788 | MSE Validação: 0.002997
Época [40/50] | MSE Treino: 0.003623 | MSE Validação: 0.002826
Época [45/50] | MSE Treino: 0.003465 | MSE Validação: 0.002676
Época [50/50] | MSE Treino: 0.003309 | MSE Validação: 0.002601

Treinamento e validação concluídos!


In [24]:
'''
# --- 1. Inferência em Lotes Sequenciais (Evitando estouro na RTX 4060) ---
lstm.eval()
y_pred_lista = []
BATCH_TEST = 512

# Convertemos os dados escalonados de teste para tensor
#x_test_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)

x_val_tensor = torch.tensor(x_val_scaled, dtype=torch.float32)

torch.cuda.empty_cache() # Garante VRAM limpa

with torch.no_grad():
    for i in range(0, len(x_val_tensor), BATCH_TEST):
        lote_x = x_val_tensor[i:i + BATCH_TEST].to(device)
        pred_lote = lstm(lote_x)
        y_pred_lista.append(pred_lote.cpu().numpy())

# Consolida a matriz predita no formato original [N_teste, 3, 1]
y_pred_scaled = np.vstack(y_pred_lista)

# --- 2. Desnormalização Tridimensional Apropriada ---
N_t, T_t, F_t = y_pred_scaled.shape

y_val_real_flat = y_val_scaled.reshape(-1, F_t)
y_val_pred_flat = y_pred_scaled.reshape(-1, F_t)

y_val_real_unidade_flat = scaler_y.inverse_transform(y_val_real_flat)
y_val_pred_unidade_flat = scaler_y.inverse_transform(y_val_pred_flat)

y_val_real_unidade = y_val_real_unidade_flat.reshape(N_t, T_t, F_t)
y_val_pred_unidade = y_val_pred_unidade_flat.reshape(N_t, T_t, F_t)

y_real_inicio = y_val_real_unidade[:, 0, 0]
y_pred_inicio = y_val_pred_unidade[:, 0, 0]

y_real_fim    = y_val_real_unidade[:, -1, 0]
y_pred_fim    = y_val_pred_unidade[:, -1, 0]

print("Vetores cronológicos extraídos com sucesso!")
print(f"Total de sequências avaliadas no bloco de teste: {len(y_real_fim)}")

# --- 4. Construção do Painel de Gráficos (Matplotlib) ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Subplot 1: Primeiro Passo de Tempo da Janela
ax1.plot(y_real_inicio[:300], label='Power Real (Início da Janela)', color='black', linewidth=1.5)
ax1.plot(y_pred_inicio[:300], label='Power Predito LSTM (Início da Janela)', color='blue', linestyle='--', linewidth=1.5)
ax1.set_title('Predição no Passo Inicial da Sequência - Primeiras 300 amostras')
ax1.set_ylabel('Potência [Watts]')
ax1.legend(frameon=True)
ax1.grid(True, linestyle='--', alpha=0.5)

# Subplot 2: Último Passo de Tempo da Janela (Refinado)
ax1.set_xlabel('') # Remove o rótulo do eixo X superior para não sobrepor
ax2.plot(y_real_fim[:300], label='Power Real (Fim da Janela)', color='black', linewidth=1.5)
ax2.plot(y_pred_fim[:300], label='Power Predito LSTM (Fim da Janela)', color='red', linestyle='--', linewidth=1.5)
ax2.set_title('Predição no Passo Final da Sequência (Com Contexto Histórico) - Primeiras 300 amostras')
ax2.set_xlabel('Amostras Cronológicas Ordenadas (Conjunto de Teste)')
ax2.set_ylabel('Potência [Watts]')
ax2.legend(frameon=True)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()'''

'\n# --- 1. Inferência em Lotes Sequenciais (Evitando estouro na RTX 4060) ---\nlstm.eval()\ny_pred_lista = []\nBATCH_TEST = 512\n\n# Convertemos os dados escalonados de teste para tensor\n#x_test_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)\n\nx_val_tensor = torch.tensor(x_val_scaled, dtype=torch.float32)\n\ntorch.cuda.empty_cache() # Garante VRAM limpa\n\nwith torch.no_grad():\n    for i in range(0, len(x_val_tensor), BATCH_TEST):\n        lote_x = x_val_tensor[i:i + BATCH_TEST].to(device)\n        pred_lote = lstm(lote_x)\n        y_pred_lista.append(pred_lote.cpu().numpy())\n\n# Consolida a matriz predita no formato original [N_teste, 3, 1]\ny_pred_scaled = np.vstack(y_pred_lista)\n\n# --- 2. Desnormalização Tridimensional Apropriada ---\nN_t, T_t, F_t = y_pred_scaled.shape\n\ny_val_real_flat = y_val_scaled.reshape(-1, F_t)\ny_val_pred_flat = y_pred_scaled.reshape(-1, F_t)\n\ny_val_real_unidade_flat = scaler_y.inverse_transform(y_val_real_flat)\ny_val_pred_unidade_flat

In [25]:
# --- Inicialização dos Estados de Memória (Mantido conforme a rede treinada) ---
num_layers = lstm.num_layers
hidden_dim = lstm.hidden_dim

# Inicializa os estados ocultos e de célula (Batch=1, Hidden_Dim=128)
h_step = torch.zeros(num_layers, 1, hidden_dim).to(device)
c_step = torch.zeros(num_layers, 1, hidden_dim).to(device)

def PredSingle(x_raw):
    """
    Recebe um vetor contendo um único instante de features (dimensão 3).
    Retorna um array NumPy contendo as 3 variáveis de saída correspondentes 
    para o instante atual, atualizando a memória recorrente interna da LSTM.
    """
    global h_step, c_step
    
    # 1. Garante o formato de array e redimensiona para o Scaler [1, num_features] (num_features = 3)
    x_arr = np.array(x_raw, dtype=np.float32).reshape(1, -1)
    
    # 2. Aplica a mesma normalização MinMaxScaler usada no treino das features
    x_scaled = scaler_X.transform(x_arr)
    
    # 3. Formata para o tensor exigido pela LSTM: [Batch=1, Time_Step=1, Features=3]
    x_tensor = torch.tensor(x_scaled, dtype=torch.float32).reshape(1, 1, -1).to(device)
    
    # 4. Executa um único passo à frente passando e coletando os estados de memória atualizados
    lstm.eval()
    with torch.no_grad():
        # Passa os estados anteriores (h_step, c_step) diretamente pela célula da LSTM
        out_lstm, (h_step, c_step) = lstm.lstm(x_tensor, (h_step, c_step))
        
        # A camada linear agora projeta os 128 neurônios para as 3 saídas (num_labels = 3)
        y_scaled_pred = lstm.linear(out_lstm)
        
        # Remove as dimensões extras do lote e do tempo para entregar ao scaler -> [1, 3]
        y_scaled_np = y_scaled_pred.cpu().numpy().reshape(1, -1)
        
    # 5. Desnormaliza as 3 saídas simultaneamente usando o scaler_y ajustado no treino
    y_real = scaler_y.inverse_transform(y_scaled_np).flatten()
    
    # Retorna o vetor contendo os 3 outputs desnormalizados na unidade original
    return y_real

In [28]:
path = r'Datasets\DEVRT\NISSAN LEAF\20230418_NISSAN_AZPEITIA_DONOSTIA_016.csv'
df1 = pd.read_csv(path)
df1.fillna(0, inplace=True)
#df1 = moving_average_df(df1, window_size=5)

pwr = (df1['Motor Pwr(w)'].values)
spd = (df1['speed'].values)
rpm = (df1['rpm'].values)
trq = (df1['Torque Nm'].values)
elv = (df1['elv_spy'].values)
lat = (df1['latitude'].values)
lon = (df1['longitude'].values)
alt = (df1['altitude'].values)
df1 = pd.DataFrame({'speed': spd, 'rpm': rpm, 'torque': trq, 'elv': elv,'lat': lat, 'lon': lon, 'alt': alt, 'power': pwr,})
#df1 = pd.DataFrame({'speed': spd, 'rpm': rpm, 'torque': trq, 'elv': elv, 'power': pwr,})
#df1 = pd.DataFrame({'speed': spd, 'rpm': rpm, 'torque': trq, 'power': pwr,})


df1 = PrepareDataframe(df1,Np)

yR,yP = [], []
for i in range(len(df1)):
    x = df1.iloc[i,:-(Np+1)].values
    yr = df1.iloc[i,-(Np+1):].values
    yp = PredSingle(x)
    yR.append(yr)
    yP.append(yp)   

yR = np.array(yR).T
yP = np.array(yP).T

In [29]:
x1 = np.arange(len(pwr))
y1 = pwr
k = 0
x2 = x1[k:-Np+k]
y2 = yP[k]
PlotSeriesPLY(ySeries=[y1,y2],xSeries=[x1,x2],w=900)